# Google Transparency Report — Complete Analysis

**Site:** https://google-transparency-report-analysis.oriz.in  
**Repo:** https://github.com/chirag127/google-transparency-report-analysis

This notebook covers all 8 Google Transparency Report datasets:

1. Copyright Removals (Web Search) — bulk CSV
2. Government Requests to Remove Content
3. Government Requests for User Information
4. HTTPS Encryption in Transit
5. Safe Browsing (Unsafe Sites)
6. Email Encryption in Transit
7. EU Right to be Forgotten
8. Traffic Disruptions

**Run all cells top-to-bottom.** Cell 2 downloads the copyright dataset (~80 MB) on first run.


In [ ]:
import sys
sys.path.insert(0, '../src')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
matplotlib.use('Agg')  # change to 'inline' in Jupyter

%matplotlib inline

from gtra.datasets import DATASETS, DATASET_BY_ID
from gtra.download import download_dataset, DATA_DIR
from gtra.analyze import run_analysis
from gtra.build_site_data import build

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Datasets registered: {len(DATASETS)}')
print(f'Data dir: {DATA_DIR}')

## 1. Copyright Removals (Web Search)

Bulk CSV published by Google. Downloads ~80 MB zip on first run.


In [ ]:
# Download if not cached
download_dataset('copyright', DATA_DIR)

In [ ]:
# Load and inspect
csvs = list((DATA_DIR / 'copyright').glob('*.csv'))
print(f'CSVs found: {len(csvs)}')
dfs = [pd.read_csv(p, low_memory=False) for p in csvs]
df = pd.concat(dfs, ignore_index=True)
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
print(f'Shape: {df.shape}')
df.head(3)

In [ ]:
# Date parsing
date_col = next((c for c in df.columns if 'date' in c), None)
if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.dropna(subset=[date_col])
    df['year'] = df[date_col].dt.year
    print(f'Date range: {df[date_col].min().date()} to {df[date_col].max().date()}')

In [ ]:
# Top copyright owners
owner_col = next((c for c in df.columns if 'copyright_owner' in c or 'owner' in c), None)
if owner_col:
    top_owners = df[owner_col].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(10, 6))
    sns.set_theme(style='whitegrid')
    top_owners.sort_values().plot.barh(ax=ax, color='#1d4e89')
    ax.set_title('Top 15 Copyright Owners by Request Count')
    ax.set_xlabel('Requests')
    plt.tight_layout()
    plt.show()
    print('Top 5:', top_owners.head(5).to_dict())

In [ ]:
# Yearly trend
if 'year' in df.columns:
    yearly = df.groupby('year').size()
    fig, ax = plt.subplots(figsize=(10, 5))
    yearly.plot(ax=ax, marker='o', color='#2e86ab', linewidth=2)
    ax.set_title('Copyright Removal Requests per Year')
    ax.set_ylabel('Requests')
    plt.tight_layout()
    plt.show()
    print(f'Peak year: {yearly.idxmax()} ({yearly.max():,} requests)')

In [ ]:
# Top targeted domains
domain_col = next((c for c in df.columns if 'domain' in c), None)
if domain_col:
    top_domains = df[domain_col].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(10, 7))
    top_domains.sort_values().plot.barh(ax=ax, color='#c73e1d')
    ax.set_title('Top 20 Most-Targeted Domains')
    ax.set_xlabel('Requests')
    plt.tight_layout()
    plt.show()

In [ ]:
# Removal rate
url_req = next((c for c in df.columns if 'urls_requested' in c or 'requested_to_remove' in c), None)
url_rem = next((c for c in df.columns if 'urls_removed' in c or 'actually_removed' in c), None)
if url_req and url_rem:
    df[url_req] = pd.to_numeric(df[url_req], errors='coerce')
    df[url_rem] = pd.to_numeric(df[url_rem], errors='coerce')
    valid = df[[url_req, url_rem]].dropna()
    total_req = int(valid[url_req].sum())
    total_rem = int(valid[url_rem].sum())
    rate = total_rem / total_req * 100 if total_req else 0
    print(f'Total URLs requested: {total_req:,}')
    print(f'Total URLs removed:   {total_rem:,}')
    print(f'Removal rate:         {rate:.1f}%')

## 2. Government Requests to Remove Content

No bulk CSV. Data from interactive report at https://transparencyreport.google.com/government-removals/overview


In [ ]:
# Illustrative data from interactive report
gov_removal_data = {
    'H1 2011': 1020, 'H2 2011': 1063, 'H1 2012': 1471, 'H2 2012': 2285,
    'H1 2013': 3330, 'H2 2013': 3846, 'H1 2014': 4823, 'H2 2014': 5169,
    'H1 2015': 6440, 'H2 2015': 6939, 'H1 2016': 7523, 'H2 2016': 7875,
    'H1 2017': 8547, 'H2 2017': 9024, 'H1 2018': 9696, 'H2 2018': 10189,
    'H1 2023': 12400
}
fig, ax = plt.subplots(figsize=(12, 5))
pd.Series(gov_removal_data).plot.bar(ax=ax, color='#1d4e89', rot=45)
ax.set_title('Government Content Removal Requests to Google')
ax.set_ylabel('Requests')
plt.tight_layout()
plt.show()
print('Source: https://transparencyreport.google.com/government-removals/overview')

## 3–8. Other Datasets

For Government User Data, HTTPS, Safe Browsing, Email Encryption, EU RTBF, and Traffic Disruptions — no bulk CSVs are published. See the stub findings from the analysis package.


In [ ]:
from gtra.analyze import analyze_stub
from gtra.datasets import DATASETS

for ds in DATASETS:
    if not ds['bulk_available']:
        result = analyze_stub(ds['id'])
        print(f"\n=== {ds['name']} ===")
        print(f"  Source: {ds['source_url']}")
        if 'key_facts' in result:
            for fact in result['key_facts']:
                print(f'  - {fact}')

## Export site data

Run this cell to export all findings as JSON to `docs/data/` for the static site.


In [ ]:
from gtra.build_site_data import build
build(DATA_DIR, OUTPUT_DIR)
print('Site data exported to docs/data/')